# IMPORT LIBRARY

In [1]:
import datetime
import mysql.connector
from mysql.connector import Error
from mysql.connector import errorcode
import time

from pytz import timezone

import pandas as pd
import numpy as np
from datetime import datetime, timedelta, date

import importlib, importlib.util

import warnings
warnings.filterwarnings("ignore")

from sqlalchemy import create_engine
import psycopg2

import requests
import json

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec

import pymysql
pd.set_option('display.max_columns', None)

# CONFIG DB

In [2]:
# Import Config Data
import importlib, importlib.util

def module_from_file(module_name, file_path):
    spec = importlib.util.spec_from_file_location(module_name, file_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

config_dl_dwh = module_from_file("config_dw", "../db_config/config_db_datawarehouse.py")

In [3]:
def dl_dwh_db_conn(db_name):
    return pymysql.connect(
        host=config_dl_dwh.hostname,
        db=db_name,
        user=config_dl_dwh.username,
        password=config_dl_dwh.password,
        port=3306
    )

# RULES / CONFIG PERHITUNGAN

Semua rules besaran komisi per payment channel ditaruh di sini (bukan di query), biar gampang diubah tanpa nyentuh SQL.

In [4]:
# outlet yang mau dihitung uang masuknya
OUTLET_CODES = ['00570', '00989']

# mapping raw payment_type_name (dari DB) -> nama channel yang dipakai di perhitungan
# NOTE: value di bawah dipakai langsung sebagai nama kolom di table, jadi ditulis snake_case
PAYMENT_TYPE_TO_CHANNEL = {
    'Cash': 'cash',
    'BRI': 'bri',
    'Go Food': 'go_food',
    'Grab Food': 'grab_food',
    'Grab Dine Out': 'grab_dine_out',
    'Shopee Food': 'shopee_food',
    'Tokopedia Go': 'tokopedia_go',
    'Jiwa+': 'jiwaplus',
    'iSellerpay (Sejutajiwariders)': 'iseller_qris',
    'iSeller Pay QRIS': 'iseller_qris',
    'QPON': 'qpon',
}
NULL_PAYMENT_CHANNEL = 'null_payment'
OTHER_PAYMENT_CHANNEL = 'other'

# besaran commission cost per channel (pct dari subtotal / net sales)
COMMISSION_PCT = {
    'cash': 0.0,
    'bri': 0.0,          # all banks
    'go_food': 0.16,
    'grab_food': 0.15,
    'grab_dine_out': 0.15,
    'shopee_food': 0.15,
    'tokopedia_go': 0.07,
    'jiwaplus': 0.0,
    'iseller_qris': 0.007,
    'qpon': 0.05,
}

# null_payment & other sengaja gak dibikin kolom per-channel sendiri (tetep kehitung
# di net_sales/tax_amount/total_sales overall karena itu diagregasi dari semua baris,
# tapi gak perlu ditampilin/di-dump sebagai kolom channel terpisah)
CHANNEL_ORDER = ['cash', 'bri', 'go_food', 'grab_food', 'grab_dine_out', 'shopee_food',
                 'tokopedia_go', 'jiwaplus', 'iseller_qris', 'qpon']

# FUNCTION

In [5]:
def get_from_dl_dwh():
    connection = dl_dwh_db_conn(config_dl_dwh.db_resource)
    cursor = connection.cursor()

    outlet_list = "', '".join(OUTLET_CODES)

    sql = f"""
        SELECT
            DATE_FORMAT(transaction_date, '%Y-%m-%d') AS date,
            outlet_code,
            outlet_name,
            transactions_payment_type_name AS payment_type,
            SUM(total_amount) AS total_amount,
            SUM(total_tax_amount) AS total_tax_amount,
            SUM(subtotal) AS subtotal
        FROM transactions_iseller_pusat
        WHERE DATE_FORMAT(transaction_date, '%Y-%m-%d') >= '{start_date_str}'
            AND DATE_FORMAT(transaction_date, '%Y-%m-%d') <= '{end_date_str}'
            AND outlet_code IN ('{outlet_list}')
            AND fulfillment_status = 'fulfilled'
            AND status = 'completed'
            AND payment_status = 'paid'
            AND transactions_status = 'success'
        GROUP BY 1, 2, 3, 4
        ORDER BY 1, 2
    """

    cursor.execute(sql)

    record = cursor.fetchall()
    columns = [desc[0] for desc in cursor.description]
    df = pd.DataFrame(record, columns=columns)

    cursor.close()
    connection.close()

    return df

In [6]:
def map_channel(payment_type):
    if pd.isna(payment_type):
        return NULL_PAYMENT_CHANNEL
    return PAYMENT_TYPE_TO_CHANNEL.get(payment_type, OTHER_PAYMENT_CHANNEL)

# CONFIG DATE RANGE

Default: jalan tiap hari proses H-1 (`yesterday`). Kalau mau custom range (backfill/reprocess), set `USE_CUSTOM_DATE_RANGE = True` lalu isi `start_date` & `end_date`.

In [7]:
USE_CUSTOM_DATE_RANGE = True  # set True kalau mau pakai start_date & end_date custom di bawah

yesterday = date.today() - timedelta(days=1)

start_date = date(2026, 9, 1)
end_date   = date(2026, 9, 21)

if USE_CUSTOM_DATE_RANGE:
    run_start_date = start_date
    run_end_date = end_date
else:
    run_start_date = yesterday
    run_end_date = yesterday

start_date_str = run_start_date.strftime('%Y-%m-%d')
end_date_str   = run_end_date.strftime('%Y-%m-%d')

print("start_date:", start_date_str)
print("end_date  :", end_date_str)

start_date: 2026-09-01
end_date  : 2026-09-21


# MAIN PROGRESS

In [8]:
df_raw = get_from_dl_dwh()
df_raw.head()

,date,outlet_code,outlet_name,payment_type,total_amount,total_tax_amount,subtotal
0,2026-09-01,00570,00570 DISTRICT 2 MEIKARTA CKR,Cash,133000.0,12090.9089,120909.090909
1,2026-09-01,00570,00570 DISTRICT 2 MEIKARTA CKR,Go Food,134700.0,12304.0000,122397.215397
2,2026-09-01,00570,00570 DISTRICT 2 MEIKARTA CKR,Grab Food,597941.0,54591.0000,543353.407043
3,2026-09-01,00570,00570 DISTRICT 2 MEIKARTA CKR,iSeller Pay QRIS,763000.0,69420.9659,693579.033579
4,2026-09-01,00570,00570 DISTRICT 2 MEIKARTA CKR,Jiwa+,232437.0,21154.0000,211279.791717


In [9]:
df = df_raw.copy()
df['channel'] = df['payment_type'].apply(map_channel)

# agregasi per date, outlet, channel (jaga-jaga kalau ada raw payment_type yg beda tapi map ke channel sama)
grouped = (
    df.groupby(['date', 'outlet_code', 'outlet_name', 'channel'], as_index=False)
        .agg(total_amount=('total_amount', 'sum'),
            total_tax_amount=('total_tax_amount', 'sum'),
            subtotal=('subtotal', 'sum'))
)

# pivot jadi wide: satu baris per date+outlet, kolom per channel (total_amount & subtotal)
pivot_amount = grouped.pivot_table(index=['date', 'outlet_code', 'outlet_name'],
                                    columns='channel', values='total_amount',
                                    aggfunc='sum', fill_value=0)
pivot_subtotal = grouped.pivot_table(index=['date', 'outlet_code', 'outlet_name'],
                                        columns='channel', values='subtotal',
                                        aggfunc='sum', fill_value=0)

for ch in CHANNEL_ORDER:
    if ch not in pivot_amount.columns:
        pivot_amount[ch] = 0
    if ch not in pivot_subtotal.columns:
        pivot_subtotal[ch] = 0

summary = grouped.groupby(['date', 'outlet_code', 'outlet_name'], as_index=False).agg(
    net_sales=('subtotal', 'sum'),
    tax_amount=('total_tax_amount', 'sum'),
    total_sales=('total_amount', 'sum'),
).set_index(['date', 'outlet_code', 'outlet_name'])

result = summary.copy()

total_commission_cost = pd.Series(0.0, index=result.index)

# per channel cuma nyimpen: total_amount (raw), subtotal (net sales), commission
# uang masuk sengaja gak disimpan di sini -> tinggal subtotal - commission kalau butuh
for ch in CHANNEL_ORDER:
    pct = COMMISSION_PCT[ch]
    amount = pivot_amount[ch]
    sub = pivot_subtotal[ch]
    commission = (sub * pct).round(2)

    result[ch] = amount
    result[f'{ch}_subtotal'] = sub
    result[f'{ch}_commission'] = commission

    total_commission_cost = total_commission_cost + commission

result['total_commission_cost'] = total_commission_cost.round(2)
result['total_uang_masuk'] = (result['net_sales'] - result['total_commission_cost']).round(2)

result = result.reset_index().sort_values(['date', 'outlet_code'])

In [10]:
# dataframe final yg mau di-dump ke table, exclude total_uang_masuk (gaperlu masuk table)
df = result.drop(columns=['total_uang_masuk'])
df.head(6)

,date,outlet_code,outlet_name,net_sales,tax_amount,total_sales,cash,cash_subtotal,cash_commission,bri,bri_subtotal,bri_commission,go_food,go_food_subtotal,go_food_commission,grab_food,grab_food_subtotal,grab_food_commission,grab_dine_out,grab_dine_out_subtotal,grab_dine_out_commission,shopee_food,shopee_food_subtotal,shopee_food_commission,tokopedia_go,tokopedia_go_subtotal,tokopedia_go_commission,jiwaplus,jiwaplus_subtotal,jiwaplus_commission,iseller_qris,iseller_qris_subtotal,iseller_qris_commission,qpon,qpon_subtotal,qpon_commission,total_commission_cost
0,2026-09-01,00570,00570 DISTRICT 2 MEIKARTA CKR,1.922045e+06,192768.1087,2114781.0,133000.0,120909.090909,0.0,0.0,0.000000,0.0,134700.0,122397.215397,19583.55,597941.0,543353.407043,81503.01,0,0,0.0,224003.0,203526.800688,30529.02,0.0,0.000000,0.00,232437.0,211279.791717,0.0,763000.0,6.935790e+05,4855.05,29700.0,27000.000000,1350.00,137820.63
1,2026-09-01,00989,00989 SUMMARECON KARAWANG KWG,4.498395e+06,450679.2289,4949018.0,239500.0,217699.172527,0.0,0.0,0.000000,0.0,112000.0,101760.851761,16281.74,43700.0,39698.607699,5954.79,0,0,0.0,634095.0,576050.567366,86407.59,507000.0,460909.090909,32263.64,880923.0,800839.090909,0.0,2288500.0,2.080255e+06,14561.79,243300.0,221181.818182,11059.09,166528.64
2,2026-09-02,00570,00570 DISTRICT 2 MEIKARTA CKR,1.798564e+06,180482.5588,1978971.7,0.0,0.000000,0.0,0.0,0.000000,0.0,37200.0,33789.516790,5406.32,386975.0,351594.799345,52739.22,0,0,0.0,341163.7,309933.405683,46490.01,156900.0,142636.363636,9984.55,147333.0,133911.179361,0.0,874500.0,7.949713e+05,5564.80,34900.0,31727.272727,1586.36,121771.26
3,2026-09-02,00989,00989 SUMMARECON KARAWANG KWG,3.802302e+06,387065.4552,4249896.0,123500.0,112244.062244,0.0,0.0,0.000000,0.0,221400.0,201186.732187,32189.88,72688.0,66022.669943,9903.40,0,0,0.0,679910.0,617855.143452,92678.27,279600.0,193635.909091,13554.51,808798.0,735159.219157,0.0,1804000.0,1.639835e+06,11478.84,260000.0,236363.636364,11818.18,171623.08
4,2026-09-03,00570,00570 DISTRICT 2 MEIKARTA CKR,2.051542e+06,205561.9783,2257075.2,0.0,0.000000,0.0,0.0,0.000000,0.0,106710.0,96923.095823,15507.70,294500.0,267583.947584,40137.59,0,0,0.0,123559.2,112268.624328,16840.29,209080.0,190072.727273,13305.09,273926.0,249023.636364,0.0,1229500.0,1.117670e+06,7823.69,19800.0,18000.000000,900.00,94514.36
5,2026-09-03,00989,00989 SUMMARECON KARAWANG KWG,4.543723e+06,455208.9816,4998859.8,458000.0,416306.306306,0.0,80000.0,72727.272727,0.0,227550.0,206720.311220,33075.25,186889.0,169813.095823,25471.96,0,0,0.0,558554.8,507527.892694,76129.18,462700.0,420636.363636,29444.55,699866.0,636168.542802,0.0,2116500.0,1.924005e+06,13468.03,208800.0,189818.181818,9490.91,187079.88


# DUMP

In [11]:
engine = create_engine("mysql+pymysql://{user}:{pw}@{hostsname}/{db}".format(hostsname=config_dl_dwh.hostname,
                                                                             user=config_dl_dwh.username,
                                                                             pw=config_dl_dwh.password,
                                                                             db=config_dl_dwh.db_target))

df.to_sql('daily_est_income_per_store_for_landlord', con=engine, if_exists='append', index=False)

42